# Deep Research Tool - Salvage & Debug Notebook

このノートブックは、エラーや接続切れで中断した調査データをサルベージするためのツールです。

## 使い方
1. エラーが発生した後、**カーネルを再起動せずに**このノートブックを開く
2. セルを順番に実行してデータをサルベージ
3. 保存されたファイルを確認

## 1. 現在の変数一覧を確認

In [ ]:
# 現在のメモリ上の変数を確認
print("=" * 60)
print("現在の変数一覧")
print("=" * 60)

for name, obj in sorted(globals().items()):
    if not name.startswith('_'):
        obj_type = type(obj).__name__
        try:
            size = len(obj) if hasattr(obj, '__len__') else '-'
        except:
            size = '-'
        print(f"  {name:30s} | {obj_type:20s} | size: {size}")

## 2. Deep Research Tool 関連オブジェクトの検索

In [ ]:
import gc

print("=" * 60)
print("メモリ上のオブジェクト検索")
print("=" * 60)

# 検索対象のクラス名
target_classes = [
    'Researcher',
    'ResearchSession', 
    'EvidenceLocker',
    'Evidence',
    'ExtractedContent',
    'PageContent',
    'SearchResult',
    'ResearchPlan',
    'TableOfContents',
]

found_objects = {}

for obj in gc.get_objects():
    class_name = type(obj).__name__
    if class_name in target_classes:
        if class_name not in found_objects:
            found_objects[class_name] = []
        found_objects[class_name].append(obj)

print("\n検出されたオブジェクト:")
for class_name, objects in found_objects.items():
    print(f"  ✓ {class_name}: {len(objects)} instances")

if not found_objects:
    print("  ✗ Deep Research Tool関連のオブジェクトが見つかりませんでした")
    print("    → カーネルが再起動されている可能性があります")

## 3. 完全サルベージ実行

In [ ]:
import gc
import json
from pathlib import Path
from datetime import datetime

def salvage_research_data(output_dir: str = None):
    """メモリ上のdeep_research_toolデータをサルベージ"""
    
    if output_dir is None:
        output_dir = f"./salvage_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    
    salvage_dir = Path(output_dir)
    salvage_dir.mkdir(exist_ok=True)
    
    print("=" * 60)
    print(f"サルベージ開始: {salvage_dir}")
    print("=" * 60)
    
    found_anything = False
    results = {}
    
    # 1. ResearchSession
    sessions = [obj for obj in gc.get_objects() 
                if type(obj).__name__ == 'ResearchSession']
    if sessions:
        found_anything = True
        results['sessions'] = sessions
        for i, s in enumerate(sessions):
            try:
                filepath = salvage_dir / f"session_{i}.json"
                with open(filepath, "w", encoding="utf-8") as f:
                    json.dump(s.to_dict(), f, ensure_ascii=False, indent=2)
                print(f"✓ Session saved: {filepath}")
                print(f"  - ID: {s.session_id}")
                print(f"  - Query: {s.query}")
                print(f"  - State: {s.state}")
                print(f"  - Section Contents: {len(s.section_contents)} sections")
            except Exception as e:
                print(f"✗ Session {i} error: {e}")
    
    # 2. EvidenceLocker
    lockers = [obj for obj in gc.get_objects() 
               if type(obj).__name__ == 'EvidenceLocker']
    if lockers:
        found_anything = True
        results['lockers'] = lockers
        for i, locker in enumerate(lockers):
            try:
                evidence_count = len(locker.get_all_evidence())
                locker.export_to_json(salvage_dir / f"evidence_{i}.json")
                locker.export_to_csv(salvage_dir / f"evidence_{i}.csv")
                print(f"✓ EvidenceLocker saved: evidence_{i}.json/csv")
                print(f"  - Evidence count: {evidence_count}")
            except Exception as e:
                print(f"✗ EvidenceLocker {i} error: {e}")
    
    # 3. Researcher (sessionとevidenceを内包)
    researchers = [obj for obj in gc.get_objects() 
                   if type(obj).__name__ == 'Researcher']
    if researchers:
        found_anything = True
        results['researchers'] = researchers
        for i, r in enumerate(researchers):
            try:
                print(f"\n✓ Researcher {i} found")
                if hasattr(r, 'session') and r.session:
                    filepath = salvage_dir / f"researcher_{i}_session.json"
                    with open(filepath, "w", encoding="utf-8") as f:
                        json.dump(r.session.to_dict(), f, ensure_ascii=False, indent=2)
                    print(f"  - Session saved: {filepath}")
                
                if hasattr(r, 'evidence_locker') and r.evidence_locker:
                    r.evidence_locker.export_to_json(salvage_dir / f"researcher_{i}_evidence.json")
                    r.evidence_locker.export_to_csv(salvage_dir / f"researcher_{i}_evidence.csv")
                    print(f"  - Evidence saved")
            except Exception as e:
                print(f"✗ Researcher {i} error: {e}")
    
    # 4. ExtractedContent
    contents = [obj for obj in gc.get_objects() 
                if type(obj).__name__ == 'ExtractedContent']
    if contents:
        found_anything = True
        results['contents'] = contents
        data = []
        for c in contents:
            try:
                data.append({
                    "source_url": getattr(c, 'source_url', ''),
                    "source_title": getattr(c, 'source_title', ''),
                    "processed_content": getattr(c, 'processed_content', ''),
                    "key_points": getattr(c, 'key_points', []),
                    "relevance_score": getattr(c, 'relevance_score', 0),
                })
            except:
                pass
        
        with open(salvage_dir / "extracted_contents.json", "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)
        print(f"\n✓ ExtractedContent saved: {len(data)} items")
    
    # 5. PageContent (生のコンテンツ)
    pages = [obj for obj in gc.get_objects() 
             if type(obj).__name__ == 'PageContent']
    if pages:
        found_anything = True
        results['pages'] = pages
        data = []
        for p in pages:
            try:
                data.append({
                    "url": getattr(p, 'url', ''),
                    "title": getattr(p, 'title', ''),
                    "text_content": getattr(p, 'text_content', '')[:10000],
                    "metadata": getattr(p, 'metadata', {}),
                })
            except:
                pass
        
        with open(salvage_dir / "page_contents.json", "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)
        print(f"✓ PageContent saved: {len(data)} items")
    
    # 6. SearchResult
    search_results = [obj for obj in gc.get_objects() 
                      if type(obj).__name__ == 'SearchResult']
    if search_results:
        found_anything = True
        results['search_results'] = search_results
        data = []
        for r in search_results:
            try:
                data.append({
                    "url": getattr(r, 'url', ''),
                    "title": getattr(r, 'title', ''),
                    "snippet": getattr(r, 'snippet', ''),
                })
            except:
                pass
        
        with open(salvage_dir / "search_results.json", "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)
        print(f"✓ SearchResult saved: {len(data)} items")
    
    # 7. Evidence (個別)
    evidences = [obj for obj in gc.get_objects() 
                 if type(obj).__name__ == 'Evidence']
    if evidences:
        found_anything = True
        results['evidences'] = evidences
        data = []
        for e in evidences:
            try:
                data.append(e.to_dict())
            except:
                try:
                    data.append({
                        "url": getattr(e, 'url', ''),
                        "title": getattr(e, 'title', ''),
                        "content_excerpt": getattr(e, 'content_excerpt', ''),
                        "relevance_score": getattr(e, 'relevance_score', 0),
                    })
                except:
                    pass
        
        with open(salvage_dir / "evidences.json", "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)
        print(f"✓ Evidence saved: {len(data)} items")
    
    print("\n" + "=" * 60)
    if found_anything:
        print(f"サルベージ完了: {salvage_dir}")
        print("\n保存されたファイル:")
        for f in sorted(salvage_dir.glob("*")):
            print(f"  - {f.name} ({f.stat().st_size:,} bytes)")
    else:
        print("サルベージ可能なデータが見つかりませんでした")
        print("Jupyterカーネルが再起動されている可能性があります")
    print("=" * 60)
    
    return results if found_anything else None

# サルベージ実行
salvaged = salvage_research_data()

## 4. サルベージ結果の確認

In [ ]:
# サルベージされたセッションの詳細確認
if salvaged and 'sessions' in salvaged:
    session = salvaged['sessions'][0]
    
    print("=" * 60)
    print("セッション詳細")
    print("=" * 60)
    print(f"Session ID: {session.session_id}")
    print(f"Query: {session.query}")
    print(f"State: {session.state}")
    print(f"Started: {session.started_at}")
    print(f"Completed: {session.completed_at}")
    
    print("\n【生成済みセクション】")
    for key, content in session.section_contents.items():
        text = content.get('content', '')
        print(f"  [{key}] {len(text):,} chars")
    
    print("\n【反復記録】")
    for it in session.iterations:
        print(f"  Section {it.section}: {it.content_extracted} contents")
else:
    print("セッションデータがありません")

In [ ]:
# サルベージされた本文の確認
if salvaged and 'sessions' in salvaged:
    session = salvaged['sessions'][0]
    
    print("=" * 60)
    print("生成済み本文")
    print("=" * 60)
    
    for key in sorted(session.section_contents.keys()):
        content = session.section_contents[key]
        text = content.get('content', '')
        
        print(f"\n### {key} ###")
        print(text[:3000])
        if len(text) > 3000:
            print(f"\n... (以下省略、全{len(text):,}文字)")
        print("-" * 40)

## 5. 既存の出力ディレクトリを確認

In [ ]:
import json
from pathlib import Path
from datetime import datetime

def check_output_directory(output_dir: str = "./output"):
    """出力ディレクトリの既存ファイルを確認"""
    
    output_path = Path(output_dir)
    
    if not output_path.exists():
        print(f"Output directory not found: {output_dir}")
        return None
    
    print("=" * 60)
    print(f"出力ディレクトリ確認: {output_dir}")
    print("=" * 60)
    
    # 全ファイル一覧
    print("\n【ファイル一覧】")
    all_files = list(output_path.rglob("*"))
    files = [f for f in all_files if f.is_file()]
    
    if not files:
        print("  ファイルがありません")
        return None
    
    for f in sorted(files, key=lambda x: x.stat().st_mtime, reverse=True):
        mtime = datetime.fromtimestamp(f.stat().st_mtime).strftime("%Y-%m-%d %H:%M:%S")
        print(f"  {mtime} | {f.relative_to(output_path)} ({f.stat().st_size:,} bytes)")
    
    return files

# 確認実行
existing_files = check_output_directory("./output")

In [ ]:
# 既存のセッションファイルを読み込む
from pathlib import Path
import json

output_dir = Path("./output")
session_files = list(output_dir.glob("session_*.json"))

if session_files:
    latest_session = max(session_files, key=lambda x: x.stat().st_mtime)
    print(f"Loading: {latest_session}")
    
    with open(latest_session, "r", encoding="utf-8") as f:
        session_data = json.load(f)
    
    print(f"\nSession ID: {session_data.get('session_id')}")
    print(f"Query: {session_data.get('query')}")
    print(f"State: {session_data.get('state')}")
    
    section_contents = session_data.get('section_contents', {})
    print(f"\n生成済みセクション: {len(section_contents)}")
    for key, content in section_contents.items():
        print(f"  [{key}] {len(content.get('content', '')):,} chars")
else:
    print("セッションファイルが見つかりません")

## 6. Jupyter 履歴から復元

In [ ]:
# Jupyterの出力履歴を確認
print("=== 出力履歴 (Out) ===")
if '_oh' in dir():
    for key, value in sorted(_oh.items()):
        value_type = type(value).__name__
        print(f"  Out[{key}]: {value_type}")
        
        # deep_research_tool関連なら詳細表示
        if value_type in ['ResearchSession', 'EvidenceLocker', 'dict']:
            print(f"    → 復元可能な可能性あり")
else:
    print("  出力履歴がありません")

print("\n=== 入力履歴 (最新10件) ===")
if '_ih' in dir():
    for i, cmd in enumerate(_ih[-10:]):
        idx = len(_ih) - 10 + i
        preview = cmd[:80].replace('\n', ' ')
        print(f"  In[{idx}]: {preview}...")
else:
    print("  入力履歴がありません")

## 7. 手動復元用ユーティリティ

In [ ]:
# 特定の変数名でオブジェクトを探す
def find_variable(name: str):
    """グローバル変数から特定の名前を探す"""
    if name in globals():
        obj = globals()[name]
        print(f"Found: {name}")
        print(f"  Type: {type(obj).__name__}")
        return obj
    else:
        print(f"Not found: {name}")
        return None

# よくある変数名を検索
common_names = ['researcher', 'session', 'result', 'evidence_locker', 'locker', 'searcher', 'client']
print("=== 一般的な変数名の検索 ===")
for name in common_names:
    find_variable(name)

In [ ]:
# 特定のオブジェクトからデータを抽出
def extract_data_from_object(obj, save_path: str = None):
    """オブジェクトからデータを抽出して保存"""
    import json
    
    obj_type = type(obj).__name__
    print(f"Extracting from: {obj_type}")
    
    data = None
    
    if hasattr(obj, 'to_dict'):
        data = obj.to_dict()
    elif isinstance(obj, dict):
        data = obj
    elif hasattr(obj, '__dict__'):
        data = {k: str(v)[:1000] for k, v in obj.__dict__.items()}
    
    if data and save_path:
        with open(save_path, 'w', encoding='utf-8') as f:
            json.dump(data, f, ensure_ascii=False, indent=2, default=str)
        print(f"Saved to: {save_path}")
    
    return data

# 使用例:
# extract_data_from_object(researcher.session, "manual_salvage.json")

---

## 補足: 今後のエラー対策

次回から以下を実行前に設定すると、途中経過が自動保存されます:

```python
from deep_research_tool import run_research

result = run_research(
    query="テーマ",
    verbose=True,  # 進捗表示
    output_dir="./output",  # 明示的に出力先指定
)
```